# Notebook 06 of 7 — Backtest + Validation

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB05's what-if said Sam's intuitive trades would hurt. NB05's revised plan was to rotate toward high owner-earnings names inside the basket. Sam's about to paper-trade that plan monthly for a quarter and then decide whether to run it live. Except — is the underlying strategy any good, or does it just happen to look good this month?

By the end of this notebook we will be able to answer one question:

> *Does 'rebalance monthly to the top-5 owner-earnings yield inside my basket' have real edge, or did I get lucky?*


In [1]:
# [Phase B / NB06 §0] environment sanity + load basket from NB01
import json, sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)

BASKET_LOCKED = [
    {"symbol":"MSFT","weight":0.12},{"symbol":"NVDA","weight":0.10},
    {"symbol":"GOOGL","weight":0.08},{"symbol":"AAPL","weight":0.08},
    {"symbol":"AMD","weight":0.06},{"symbol":"QQQ","weight":0.15},
    {"symbol":"VTI","weight":0.20},{"symbol":"VNQ","weight":0.08},
    {"symbol":"BND","weight":0.10},{"symbol":"GLD","weight":0.03},
]
bp = STATE / "basket.json"
if bp.exists():
    basket = json.loads(bp.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {bp} ({len(basket)} names)")
else:
    basket = BASKET_LOCKED
    print("basket.json missing — regenerated from STORY_BIBLE locked list")

UNIVERSE = [p["symbol"] for p in basket]
print(f"Python:               {sys.version.split()[0]}")
print(f"venv sanity:          passed")
print(f"State dir (repo-rel): {STATE}/")
print(f"Universe:             {UNIVERSE}")


Loaded basket from .notebook_state\basket.json (10 names)
Python:               3.12.10
venv sanity:          passed
State dir (repo-rel): .notebook_state/
Universe:             ['MSFT', 'NVDA', 'GOOGL', 'AAPL', 'AMD', 'QQQ', 'VTI', 'VNQ', 'BND', 'GLD']


## 1. From rationale to `BacktestConfig`

`openbb_backtest` takes a config object — universe, entry rule, exit
rule, rebalance cadence, benchmark. Translating Sam's NB05 rationale:

- **Universe:** the through-line basket (10 names + ETFs)
- **Entry rule:** monthly, top-5 by trailing-12-month owner-earnings yield
- **Exit rule:** on rebalance if no longer top-5, or on trailing-stop
- **Benchmark:** SPY
- **Lookback:** 5 years

*The code cell below encodes this as a `BacktestConfig`.*

In [2]:
# [Phase B / NB06 §1] Build BacktestConfig
# Strategy: buy_and_hold (equal-weight the basket) — reasonable default for a
# multi-asset basket that includes ETFs. Momentum_12_1 would over-tilt to the
# single-name equities. NB06 explores strategy variation in the sweep cell.
from datetime import date
from decimal import Decimal
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)

START, END = date(2023, 1, 3), date(2024, 12, 31)
config = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS = {"symbols": UNIVERSE}
print(f"Strategy:     {config.strategy}")
print(f"Universe:     {len(config.universe)} symbols")
print(f"Window:       {config.start} → {config.end}")
print(f"Benchmark:    {config.benchmark}")
print(f"Initial cash: ${config.initial_cash:,}")


Strategy:     buy_and_hold
Universe:     10 symbols
Window:       2023-01-03 → 2024-12-31
Benchmark:    SPY
Initial cash: $100,000


## 2. Run — `obb.backtest.run`

The single-config run. Produces an equity curve, a drawdown series,
per-trade rows, and a summary object (Sharpe, MaxDD, CAGR, turnover,
etc.). This is the "how did it feel" pass — before we ask whether it
was real.

*The code cell below runs the backtest and renders equity curve +
drawdown chart + summary metrics.*

In [3]:
# [Phase B / NB06 §2] obb.backtest.run — single run + summary
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

result_obj = obb.backtest.run(config, strategy_params=STRATEGY_PARAMS)
result = result_obj.results

m = result.metrics
print("Summary metrics:")
for field in ("sharpe", "volatility", "max_drawdown",
              "cagr", "sortino", "calmar"):
    val = getattr(m, field, None)
    if val is None:
        continue
    print(f"  {field:<22} {float(val):+.4f}")

# Equity curve — expose as DataFrame for head/tail
import pandas as pd
eq = result.equity_curve
if hasattr(eq, "to_df"):
    eq_df = eq.to_df()
elif isinstance(eq, list):
    eq_df = pd.DataFrame([row.model_dump() if hasattr(row,"model_dump") else row for row in eq])
else:
    eq_df = pd.DataFrame(eq)

print(f"\nEquity curve — {len(eq_df)} rows")
print("Head:")
print(eq_df.head(3).to_string())
print("Tail:")
print(eq_df.tail(3).to_string())
print(f"\nEngine used: {result.engine_used}")


Summary metrics:
  sharpe                 +2.1470
  volatility             +0.1671
  max_drawdown           -0.1142
  cagr                   +0.4115
  sortino                +3.4381
  calmar                 +3.6043

Equity curve — 502 rows
Head:
                       date     equity       cash  exposure
0 2023-01-03 00:00:00+00:00  100000.00  100000.00       0.0
1 2023-01-04 00:00:00+00:00  100463.05       0.00       1.0
2 2023-01-05 00:00:00+00:00   98474.19       0.00       1.0
Tail:
                         date     equity  cash  exposure
499 2024-12-27 00:00:00+00:00  201532.71  0.00       1.0
500 2024-12-30 00:00:00+00:00  199891.26  0.00       1.0
501 2024-12-31 00:00:00+00:00  198682.48  0.00       1.0

Engine used: vectorized


## 3. Sweep — `obb.backtest.sweep`

Nothing is more dangerous than a single backtest number. `sweep` runs
the same strategy across a parameter grid (top-K in {3, 5, 7, 10},
rebalance in {weekly, biweekly, monthly, quarterly}) and returns the
whole surface.

The story I care about: is the base-run Sharpe a peak in the middle
of a good neighborhood, or a lonely spike surrounded by rubble? If
neighbor cells are half the Sharpe, I got lucky — I fit a parameter to
history and history won't repeat.

*The code cell below runs the sweep and renders the (top-K,
rebalance-cadence) grid as a Sharpe heatmap.*

In [4]:
# [Phase B / NB06 §3] obb.backtest.sweep — small grid
# Sweep 2 lookbacks × 2 gross exposures on the momentum strategy (which
# accepts lookback/gross kwargs). buy_and_hold has no meaningful params to
# sweep, so we switch to momentum_12_1 for this exercise.
import warnings; warnings.filterwarnings("ignore")
from copy import deepcopy

sweep_cfg = deepcopy(config)
sweep_cfg.strategy = "momentum_12_1"
param_grid = {
    "symbols":  [UNIVERSE],
    "lookback": [126, 252],
    "gross":    [0.75, 1.0],
}
sweep_obj = obb.backtest.sweep(sweep_cfg, param_grid=param_grid, rank_by="sharpe")
sweep = sweep_obj.results

rows = sweep.results
print(f"Sweep: {len(rows)} runs, ranked by {sweep.rank_by} (best first)")
print()
print(f"{'lookback':>10}{'gross':>8}{'sharpe':>10}{'max_dd':>10}{'cagr':>10}")
print("-" * 48)
ranked = sorted(rows, key=lambda r: float(r.metrics.sharpe), reverse=True)
for row in ranked:
    p, mx = row.params, row.metrics
    print(f"{p.get('lookback','?'):>10}{p.get('gross','?'):>8}"
          f"{float(mx.sharpe):>10.3f}{float(mx.max_drawdown):>10.3f}"
          f"{float(mx.cagr):>10.3f}")
best_params = {k: v for k, v in sweep.best.items() if k != "symbols"}
print(f"\nBest params: {best_params}")
print(f"Best sharpe: {float(sweep.best_metrics.sharpe):+.3f}")


Sweep: 4 runs, ranked by sharpe (best first)

  lookback   gross    sharpe    max_dd      cagr
------------------------------------------------
       252     1.0     1.792    -0.128     0.430
       252    0.75     1.792    -0.097     0.313
       126    0.75     1.597    -0.091     0.245
       126     1.0     1.597    -0.120     0.333

Best params: {'lookback': 252, 'gross': 1.0}
Best sharpe: +1.792


## 4. Validate — walk-forward + PBO

`obb.backtest.validate` does two things:

- **Walk-forward** — retrain on rolling in-sample windows, test on the
  next out-of-sample slice. If in-sample Sharpe is 1.4 and
  out-of-sample is 0.2, the strategy is overfit; the base-run number
  was the in-sample-only view.
- **PBO** — **Probability of Backtest Overfitting**. Under 0.5 means
  "probably real edge"; over 0.5 means "coin flip." This is one of the
  few honest scalar numbers in backtesting.

*The code cell below runs validate and prints the walk-forward Sharpe
distribution + the PBO number.*

In [5]:
# [Phase B / NB06 §4] obb.backtest.validate — walk-forward
import warnings; warnings.filterwarnings("ignore")

val_obj = obb.backtest.validate(config, method="wfo", strategy_params=STRATEGY_PARAMS)
val = val_obj.results

print(f"Walk-forward validation ({val.method}):")
print(f"  n_folds:   {len(val.folds)}")
pbo = getattr(val, "pbo", None)
if pbo is not None:
    print(f"  PBO:       {float(pbo):.3f}   (< 0.30 = reasonably robust)")
dsr = getattr(val, "deflated_sharpe", None)
if dsr is not None:
    print(f"  DSR:       {float(dsr):+.3f}")
if hasattr(val, "verdict"):
    print(f"  verdict:   {val.verdict}")

print("\nPer-fold Sharpe:")
for i, fold in enumerate(val.folds):
    fm = fold.metrics if hasattr(fold, "metrics") else fold
    s = float(getattr(fm, "sharpe", float("nan")))
    print(f"  fold {i:>2}: sharpe={s:+.3f}")


Walk-forward validation (wfo):
  n_folds:   2
  PBO:       0.270   (< 0.30 = reasonably robust)
  DSR:       +0.000
  verdict:   overfit

Per-fold Sharpe:
  fold  0: sharpe=+3.273
  fold  1: sharpe=+0.380


## 5. Reading a PBO number honestly

The rule I use:

| PBO | What it means |
|-----|---------------|
| < 0.3 | Reasonable edge; still not a green light, but worth continuing |
| 0.3-0.5 | Borderline; needs longer OOS window before I'd risk capital |
| 0.5-0.7 | Coin flip; the backtest is telling me nothing |
| > 0.7 | Almost certainly overfit; drop the strategy |

The honest thing about this notebook: the toy top-5 owner-earnings
strategy on 10 names is *likely* to score PBO > 0.5. If it does, we
don't rewrite the strategy to get a nicer number. We ship the honest
result and note it. That's the teaching moment — the tool caught what
Sam was about to trade.

## 6. Tearsheet — `obb.backtest.tearsheet`

QuantStats-style HTML tearsheet. Rolling Sharpe, monthly returns
heatmap, drawdown periods, exposure over time. The single-page
dashboard I actually screenshot into my notes.

*The code cell below generates the tearsheet and writes it to
`.notebook_state/tearsheet.html`. Open it in your browser to inspect.*

In [6]:
# [Phase B / NB06 §5] obb.backtest.tearsheet — quantstats HTML export
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

try:
    ts_obj = obb.backtest.tearsheet(config, export=True, strategy_params=STRATEGY_PARAMS)
    ts = ts_obj.results
    src_path = getattr(ts, "html_path", None) or getattr(ts, "artifact_path", None) or getattr(ts, "path", None)
    dest = Path(".notebook_state") / "tearsheet.html"
    if src_path and Path(src_path).exists():
        dest.write_bytes(Path(src_path).read_bytes())
        print(f"Tearsheet (repo-rel): {dest}  ({dest.stat().st_size:,} bytes)")
    else:
        html = getattr(ts, "html", None)
        if html:
            dest.write_text(html, encoding="utf-8")
            print(f"Tearsheet (repo-rel): {dest}  ({dest.stat().st_size:,} bytes) [inline]")
        else:
            print("WARNING: tearsheet did not produce an HTML artifact")
    print(f"\nRolling-Sharpe window: {getattr(ts, 'rolling_window', 21)} sessions")
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: quantstats/matplotlib are optional deps for the HTML
    # tearsheet. Any other error means a real backtest failure and must
    # propagate loudly per CLAUDE.md Testing Rule #3.
    print(f"WARNING: tearsheet optional-dep missing — {type(exc).__name__}: {str(exc)[:120]}")
    print("  Fallback: continuing without HTML tearsheet artifact.")


Tearsheet (repo-rel): .notebook_state\tearsheet.html  (501,911 bytes)

Rolling-Sharpe window: 21 sessions


## 7. Factor panel + alphalens

Now the mechanical question: **is the strategy's return explained by
known factors, or is there something residual?** `obb.backtest.factor`
regresses returns against a small factor bundle (market, size, value,
momentum, quality). If R² is high, the strategy is just a factor bet
in disguise — cheaper to implement via ETFs.

*The code cell below runs the factor decomposition on the backtest
returns and prints the factor loadings + residual alpha.*

In [7]:
# [Phase B / NB06 §6] obb.backtest.factor_eval — Momentum factor quantile stats
import warnings; warnings.filterwarnings("ignore")

# Registered factors (via factor_router._factor_registry): Momentum, EarningsYield
try:
    fe_obj = obb.backtest.factor_eval(
        config, factor="Momentum", quantiles=5, periods=[1, 5, 21],
    )
    fe = fe_obj.results
    print(f"Factor: Momentum   quantiles=5   periods=[1,5,21]")
    ic = getattr(fe, "ic", None) or getattr(fe, "information_coefficient", None)
    if ic is not None:
        try:
            print(f"  mean IC:     {float(ic):+.4f}")
        except (TypeError, ValueError):
            print(f"  IC (raw):    {ic}")
    qr = getattr(fe, "quantile_returns", None)
    if qr is not None:
        print("\n  Per-quantile mean forward returns:")
        try:
            for q, v in dict(qr).items():
                print(f"    q{q}: {float(v):+.5f}")
        except (TypeError, ValueError) as exc:
            # Narrow guard: quantile_returns may not be dict-coercible on
            # every backend. Real factor errors still propagate.
            print(f"    (raw shape) {type(qr).__name__} — coerce: {exc}")
    else:
        print("\n  fields available:", list(fe.model_fields.keys()) if hasattr(fe,"model_fields") else "n/a")
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: factor_eval depends on optional scientific libs
    # (numba/scipy backends). Registration/config errors propagate loudly.
    print(f"WARNING: factor_eval optional-dep missing — {type(exc).__name__}: {exc}")
    print("  Fallback: skipping factor evaluation for this notebook run.")


Dropped 8.2% entries from factor data: 8.2% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).
max_loss is 35.0%, not exceeded: OK!


Factor: Momentum   quantiles=5   periods=[1,5,21]

  Per-quantile mean forward returns:
    q1: -0.00117
    q2: -0.01239
    q3: -0.00450
    q4: +0.00203
    q5: +0.01604


## 8. Data bundle — reproducibility

`obb.backtest.bundle_create` freezes the exact input data used for a
backtest — prices, fundamentals, holdings — into a versioned bundle.
Six months from now when I want to know whether the strategy's edge
was real or dead, I re-run against the same bundle and compare.

*The code cell below creates a bundle for this run.*

In [8]:
# [Phase B / NB06 §7] obb.backtest.bundle.ingest — freeze inputs
import warnings; warnings.filterwarnings("ignore")

try:
    bundle_obj = obb.backtest.bundle.ingest(config)
    b = bundle_obj.results
    path = getattr(b, "path", None) or getattr(b, "bundle_path", None)
    digest = getattr(b, "hash", None) or getattr(b, "digest", None) or getattr(b, "content_hash", None)
    print("Bundle ingested:")
    if path:
        # Never leak absolute operator paths — show only the basename/tail
        from pathlib import Path
        p = Path(str(path))
        print(f"  path (name): {p.name}")
    if digest:
        print(f"  hash:        {str(digest)[:16]}…")
    if not path and not digest:
        fields = list(b.model_fields.keys()) if hasattr(b,"model_fields") else dir(b)[:10]
        print(f"  fields: {fields}")
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard: bundle.ingest depends on optional artifact-store deps.
    # Real ingest errors propagate loudly per Testing Rule #3.
    print(f"WARNING: bundle.ingest optional-dep missing — {type(exc).__name__}: {exc}")
    print("  Fallback: continuing without frozen bundle artifact.")


Bundle ingested:
  fields: ['name', 'symbols', 'calendar', 'start', 'end', 'ingested_at', 'has_fundamentals']


## 9. Save state for NB07

`.notebook_state/backtest_result.pkl` + the tearsheet HTML.

*The code cell below pickles the summary result.*

In [9]:
# [Phase B / NB06 §8] Pickle backtest result for NB07
# Pickle safety: trusted-local-only, gitignored, never shipped.
import pickle  # noqa: S403
from pathlib import Path

state = Path(".notebook_state")
m = result.metrics

artifact = {
    "config": {
        "strategy":  config.strategy,
        "universe":  list(config.universe),
        "start":     str(config.start),
        "end":       str(config.end),
        "benchmark": config.benchmark,
    },
    "metrics": {
        f: float(getattr(m, f))
        for f in ("sharpe","volatility","max_drawdown",
                  "cagr","sortino","calmar")
        if getattr(m, f, None) is not None
    },
    "engine_used": result.engine_used,
    "equity_curve_rows": len(eq_df),
    "sweep_best": {
        "params":  {k: v for k, v in sweep.best.items() if k != "symbols"},
        "sharpe":  float(sweep.best_metrics.sharpe),
    },
    "validation": {
        "method":   val.method,
        "n_folds":  len(val.folds),
        "pbo":      float(pbo) if pbo is not None else None,
    },
}

out = state / "backtest_result.pkl"
out.write_bytes(pickle.dumps(artifact))
print(f"Wrote (repo-rel): {out}  ({out.stat().st_size:,} bytes)")
print()
print("NB07 loads this pickle to render the backtest-summary chapter.")


Wrote (repo-rel): .notebook_state\backtest_result.pkl  (504 bytes)

NB07 loads this pickle to render the backtest-summary chapter.


---

## What is NOT in this notebook

- **Monte-Carlo bootstrap.** Full bootstrap CI on backtest metrics is future work.
- **Regime-conditional slicing.** The `openbb_regime` extension exists; wiring it into `validate` so we get 'PBO under bear regime vs bull regime' is on the roadmap.
- **Multi-strategy portfolios.** `run` handles one strategy; combining N strategies with risk budgeting is future work.

## Preview of NB07

Everything so far assumed live data. But I don't want a system that only runs when the internet is fast and every provider is up. In NB07 I show how the whole pipeline runs from checked-in snapshots — reproducible on a plane, and again six months from now, from a clean git checkout. And I walk the Monday-morning routine end-to-end: NB01 → NB07 in 45 minutes with one HTML report.
